# Exon Inclusion Analysis

In [1]:
!pip install ipykernel
!python -m ipykernel install --user --name finalproject --display-name "Python (final project)"

Installed kernelspec finalproject in /home/biouser/.local/share/jupyter/kernels/finalproject


In [10]:
## 0. Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from scipy.stats import spearmanr, bootstrap
from gtfparse import read_gtf
import pysam  
from matplotlib.colors import LinearSegmentedColormap
from functions import (
    load_type_neurons
)

In [3]:
type_alt = pd.read_table('../../data/altExonUsage_devel_type.gz', sep='\t', compression='gzip')
type_alt.shape

(8962, 34)

In [8]:
type_alt.tail(15)

,Exon,Gene,P14_Hippocampus_Astro,P14_Hippocampus_Oligo,P14_Hippocampus_ExciteNeuron,P14_Hippocampus_InhibNeuron,P14_VisCortex_Astro,P14_VisCortex_Oligo,P14_VisCortex_ExciteNeuron,P14_VisCortex_InhibNeuron,...,P28_VisCortex_ExciteNeuron,P28_VisCortex_InhibNeuron,P56_Hippocampus_Astro,P56_Hippocampus_Oligo,P56_Hippocampus_ExciteNeuron,P56_Hippocampus_InhibNeuron,P56_VisCortex_Astro,P56_VisCortex_Oligo,P56_VisCortex_ExciteNeuron,P56_VisCortex_InhibNeuron
8947,chr16_93597361_93597420_-,ENSMUSG00000022948.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.333333,NaN
8948,chr16_96190194_96190229_-,ENSMUSG00000045275.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,NaN
8949,chr16_14111545_14111771_-,ENSMUSG00000060657.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.935484,NaN
8950,chr17_46539253_46539388_-,ENSMUSG00000040327.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.727273,NaN
8951,chr17_34847408_34847461_-,ENSMUSG00000040356.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.727273,NaN
8952,chr17_32381084_32381156_-,ENSMUSG00000024050.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.153846,NaN
8953,chr17_56884751_56884968_+,ENSMUSG00000024209.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.411765,NaN
8954,chr17_35235488_35235558_+,ENSMUSG00000024403.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.400000,NaN
8955,chr18_34584214_34584285_-,ENSMUSG00000049357.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.333333,NaN
8956,chr18_12190960_12191023_+,ENSMUSG00000024410.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.250000,NaN


In [13]:
type_neurons = load_type_neurons("../../data/altExonUsage_devel_type.gz")
type_neurons.head(10)

,P14_Hippocampus_ExciteNeuron,P14_Hippocampus_InhibNeuron,P14_VisCortex_ExciteNeuron,P14_VisCortex_InhibNeuron,P21_Hippocampus_ExciteNeuron,P21_Hippocampus_InhibNeuron,P21_VisCortex_ExciteNeuron,P21_VisCortex_InhibNeuron,P28_Hippocampus_ExciteNeuron,P28_Hippocampus_InhibNeuron,P28_VisCortex_ExciteNeuron,P28_VisCortex_InhibNeuron,P56_Hippocampus_ExciteNeuron,P56_Hippocampus_InhibNeuron,P56_VisCortex_ExciteNeuron,P56_VisCortex_InhibNeuron
GE,,,,,,,,,,,,,,,,
chr1_162273620_162273649_-::ENSMUSG00000040265.16,0.032520,0.000000,0.008152,0.000000,0.049351,0.024000,0.000000,0.000000,0.015544,0.007812,0.001163,0.000000,0.015831,0.026316,0.005181,0.001821
chr1_186690721_186690804_-::ENSMUSG00000039239.14,0.000000,NaN,NaN,NaN,0.075472,NaN,NaN,NaN,0.083333,NaN,NaN,NaN,0.139073,NaN,0.019608,0.125000
chr1_172286198_172286396_-::ENSMUSG00000007097.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.818182,NaN
chr1_83384894_83384972_-::ENSMUSG00000026163.17,0.608364,0.642140,0.707604,0.657718,0.631307,0.707113,0.767779,0.776471,0.670919,0.666667,0.789824,0.805970,0.641826,0.694915,0.781662,0.712150
chr1_172279313_172279467_-::ENSMUSG00000007097.14,1.000000,NaN,1.000000,NaN,0.823529,NaN,0.823529,NaN,1.000000,NaN,1.000000,NaN,1.000000,NaN,0.965517,NaN
chr1_163253977_163254048_-::ENSMUSG00000026586.16,0.545455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chr1_172287200_172287468_-::ENSMUSG00000007097.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chr1_16486020_16486150_-::ENSMUSG00000025920.19,0.204724,0.032967,0.240229,0.097222,0.222116,0.230769,0.275465,0.102564,0.223958,0.000000,0.230177,0.132075,0.218418,0.050633,0.220895,0.145867
chr1_162675605_162675841_-::ENSMUSG00000040225.15,0.812834,0.714286,0.824022,0.848485,0.891247,0.795455,0.818493,0.914286,0.869822,1.000000,0.841040,NaN,0.950530,0.840000,0.900850,0.959184
